**Timeline**

* Установка библиотек
* Сбор данных (чеков), формирование БД на aiven.
* Подключение к БД



In [ ]:
!apt-get install tesseract-ocr -y
!apt-get install tesseract-ocr-rus -y
!pip install pytesseract pandas python-telegram-bot==20.7 opencv-python-headless psycopg2-binary fuzzywuzzy python-Levenshtein
!pip install openai --upgrade -q

print("Библиотеки установлены")

In [ ]:
import os, re, asyncio
import numpy as np
import io, logging
from PIL import Image, ImageEnhance
import pytesseract, cv2, psycopg2
from psycopg2 import pool
from google.colab import userdata

logging.basicConfig(level=logging.ERROR)
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'


In [ ]:
DB_HOST = "eco-scan-ecobot.e.aivencloud.com"
DB_PORT = "10408"
DB_NAME = "defaultdb"
DB_USER = "avnadmin"
DB_PASS = userdata.get('DB_PASSWORD')

db_Pool = psycopg2.pool.SimpleConnectionPool(1, 10, host=DB_HOST, port=DB_PORT, database=DB_NAME, user=DB_USER, password=DB_PASS, sslmode='require')
print("Подключено к БД")

def load_data():
    conn = db_Pool.getconn()
    try:
        cursor = conn.cursor()
        cursor.execute("""SELECT name, matched_product_id FROM lenta WHERE name IS NOT NULL
            UNION ALL SELECT name, matched_product_id FROM magnit WHERE name IS NOT NULL
            UNION ALL SELECT name, matched_product_id FROM okey WHERE name IS NOT NULL
            UNION ALL SELECT name, matched_product_id FROM perekrestok WHERE name IS NOT NULL
            UNION ALL SELECT name, matched_product_id FROM x5 WHERE name IS NOT NULL""")
        store_prods = {}
        for name, matched_id in cursor.fetchall():
            if name and len(name) > 3:
                store_prods[name.lower()] = matched_id

        cursor.execute("""SELECT pr.id, pr.name, c.name as category, pr.carbon_footprint FROM product_reference pr JOIN categories c ON pr.category_id = c.id""")
        products = {}
        for product_id, name, category, footprint in cursor.fetchall():
            products[product_id] = {'name': name, 'category': category, 'co2': float(footprint) if footprint else 0.0}
        print(f"Загружено названий из магазинов: {len(store_prods)}")
        print(f"Загружено продуктов из справочника: {len(products)}")
        return store_prods, products
    except:
        print("Ошибка")
        return {}, {}
    finally:
        cursor.close()
        db_Pool.putconn(conn)

store_prods, prod_info = load_data()

https://pypi.org/project/pytesseract/
https://docs.opencv.org/4.x/
https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html
https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html
https://www.psycopg.org/docs/
https://www.psycopg.org/docs/usage.html#connection-parameters
https://github.com/seatgeek/fuzzywuzzy
https://github.com/ztane/python-Levenshtein/
https://pillow.readthedocs.io/en/stable/reference/ImageEnhance.html
